# Fase 1 — Denoising de ECG (MIT-BIH)
## Projeto de Processamento de Sinais Digitais — Grupo 6

**Apresentadores:** Juliano · Pedro · Matheo  
**Data:** 28/04 · **Registo de referência:** MIT-BIH 100 (ritmo sinusal normal)

- Projeto em quatro etapas; esta apresentação cobre a **Etapa 01 — Denoising por FIR**.
- Objetivo: remover ruídos típicos de ECG sem deformar a morfologia clínica do sinal.
- Validação visual e numérica sobre os 10 segundos iniciais do Registo 100.

> 🎤 **Fala:** Juliano (abertura).

## 1. Resumo do Projeto e Objetivos

**Problema.** Sinais reais de ECG carregam três famílias de ruído que comprometem o diagnóstico:

- **Baseline wander** (< 0.5 Hz) — respiração e movimento do paciente.
- **Interferência da rede elétrica (PLI)** (≈ 60 Hz) — acoplamento da rede de energia.
- **Ruído muscular / EMG** (> 40 Hz) — contrações musculares involuntárias.

**Objetivos da Fase 1.**

- Atenuar as três famílias de ruído com **filtros FIR de fase linear**.
- Preservar a morfologia das ondas P, complexo QRS e onda T (sem deslocamento temporal).
- Quantificar a melhora com uma métrica simples de **SNR em dB**.

> 🎤 **Fala:** Juliano.

## 2. Anatomia do ECG: Ondas P, QRS e T

![Anatomia do ECG](../imgs/image.png)

Cada batimento cardíaco gera um padrão fisiológico bem definido no ECG:

- **Onda P** — despolarização (ativação elétrica) dos **átrios**. Pequena amplitude, baixa frequência.
- **Complexo QRS** — despolarização dos **ventrículos**. É o pico de maior amplitude e a marca temporal do batimento (pico R).
- **Onda T** — **repolarização** ventricular, ou seja, a recuperação elétrica do coração antes do próximo ciclo.

**Por que filtro FIR de fase linear?**

- Atraso de grupo constante: todos os componentes de frequência são deslocados pelo mesmo tempo.
- Resultado: as ondas P, QRS e T **não são deformadas** nem deslocadas relativamente entre si.
- É uma exigência clínica: medir intervalos PR, QRS e QT exige preservação morfológica.

> 🎤 **Fala:** Pedro.

## 3. Base de Dados: MIT-BIH Arrhythmia Database

- **Referência mundial** em cardiologia computacional, distribuída pela PhysioNet.
- **48 registos** de ECG de pacientes reais, ≈ 30 minutos cada, **2 canais** (MLII e V5 tipicamente).
- **Frequência de amostragem:** `fs = 360 Hz` (Nyquist = 180 Hz, suficiente para a faixa diagnóstica 0.5–40 Hz).
- Anotações `.atr` marcadas por especialistas, identificando o instante do **pico R** e o tipo de batimento.
- **Registo 100** — escolhido para a Fase 1 por apresentar **ritmo sinusal normal**, servindo de referência de "normalidade" para validar que o filtro não introduz artefatos.

> 🎤 **Fala:** Pedro.

In [ ]:
from __future__ import annotations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import wfdb

from src.config import mitdb_record_dir
from src.preprocessing.fir_filters import (
    design_highpass,
    design_bandstop,
    design_lowpass,
    design_lowpass_remez,
    apply_filtfilt,
)

RECORD_NAME = "100"
record_path = (Path(mitdb_record_dir()) / RECORD_NAME).as_posix()

record = wfdb.rdrecord(record_path)
ann = wfdb.rdann(record_path, "atr")

print("fs (Hz):", record.fs)
print("Canais:", record.sig_name)
print("Duração (s):", record.sig_len / record.fs)

## 4. Visualização do Sinal Cru — Registo 100, primeiros 10 s

- Janela de **10 segundos** (`fs · 10 = 3600 amostras`) no canal **MLII** (derivação modificada II).
- As linhas verticais marcam as **anotações WFDB** dos picos R fornecidas pelos cardiologistas.
- Observe a oscilação de baixa frequência da linha de base e o ruído fino sobreposto ao traçado.

> 🎤 **Fala:** Pedro.

In [ ]:
fs = float(record.fs)
window = int(fs * 10)
t = np.arange(window) / fs

ch = 0  # MLII
x_raw = record.p_signal[:window, ch]
ann_in_window = ann.sample[ann.sample < window]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, x_raw, color="tab:blue", linewidth=0.9, label="ECG cru (MLII)")
for s in ann_in_window:
    ax.axvline(s / fs, color="tab:red", alpha=0.3, linewidth=0.8)
ax.set_xlabel("Tempo (s)")
ax.set_ylabel("Amplitude (mV)")
ax.set_title(f"MIT-BIH {RECORD_NAME} — primeiros 10 s, canal {record.sig_name[ch]}")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right")
fig.tight_layout()
plt.show()

## 5. Estratégia de Filtragem — FIR de Fase Linear

A Fase 1 utiliza uma cascata de **três filtros FIR** projetados sobre `fs = 360 Hz`:

| Filtro | Tipo | Banda | Alvo de ruído |
|---|---|---|---|
| Passa-alta | FIR Hamming | 0.5 Hz | Baseline wander (< 0.5 Hz) |
| Rejeita-faixa | FIR Hamming | 59–61 Hz | Rede elétrica (PLI 60 Hz) |
| Passa-baixa | FIR Hamming | 40 Hz | Ruído muscular / EMG (> 40 Hz) |

Duas técnicas de projeto comparadas:

- **Janela de Hamming** — método clássico, simples e robusto; ordem ≈ `3.3 · fs / Δf`.
- **Parks-McClellan (Remez)** — projeto **equiripple** ótimo no sentido de Chebyshev; ordens menores para a mesma especificação.

Aplicação em **fase zero** com `filtfilt` (forward-backward), garantindo **atraso nulo** e preservação dos picos R.

> 🎤 **Fala:** Matheo.

In [ ]:
# Projeto dos filtros FIR (janela de Hamming)
h_hp = design_highpass(fs=fs, cutoff_hz=0.5, transition_hz=0.5)
h_bs = design_bandstop(fs=fs, low_hz=59.0, high_hz=61.0, transition_hz=1.0)
h_lp = design_lowpass(fs=fs, cutoff_hz=40.0, transition_hz=8.0)

# Variante Parks-McClellan (equiripple) para o passa-baixa
h_lp_pm = design_lowpass_remez(fs=fs, cutoff_hz=40.0, transition_hz=8.0)

print(f"FIR HP  (Hamming)        : N = {len(h_hp):4d} taps")
print(f"FIR BS  (Hamming, 60 Hz) : N = {len(h_bs):4d} taps")
print(f"FIR LP  (Hamming)        : N = {len(h_lp):4d} taps")
print(f"FIR LP  (Parks-McClellan): N = {len(h_lp_pm):4d} taps")

In [ ]:
# Pipeline completo aplicado em fase zero (filtfilt) sobre os 10 s do canal MLII
x_hp = apply_filtfilt(h_hp, x_raw)
x_bs = apply_filtfilt(h_bs, x_hp)
x_filt = apply_filtfilt(h_lp, x_bs)

## 6. Resultados — Comparação Cru vs. Filtrado no Domínio do Tempo

- **Azul:** ECG cru (entrada) — observe a oscilação da linha de base e o ruído fino.
- **Laranja:** ECG filtrado (saída do pipeline FIR cascata).
- O filtro **alinha o sinal sobre uma linha de base estável** e remove o tremor de alta frequência, **preservando a localização e amplitude relativa** das ondas P, QRS e T.

> 🎤 **Fala:** Matheo.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(t, x_raw, color="tab:blue", linewidth=0.9, label="Cru")
axes[0].set_ylabel("Amplitude (mV)")
axes[0].set_title("ECG cru — MIT-BIH 100, canal MLII")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="upper right")

axes[1].plot(t, x_raw, color="tab:blue", linewidth=0.7, alpha=0.45, label="Cru")
axes[1].plot(t, x_filt, color="tab:orange", linewidth=1.1, label="Filtrado (FIR Hamming cascata)")
axes[1].set_xlabel("Tempo (s)")
axes[1].set_ylabel("Amplitude (mV)")
axes[1].set_title("Sobreposição: cru vs. filtrado em fase zero")
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="upper right")

fig.tight_layout()
plt.show()

## 7. Avaliação Objetiva — SNR Simplificado em dB

Sem sinal limpo de referência (ground truth), aproximamos o **SNR** como a razão de potências entre a **banda diagnóstica** e a **banda de ruído**, integradas via Welch:

$$\mathrm{SNR}_{dB} = 10\,\log_{10}\!\left(\frac{P_{\text{sinal} \,(1\text{–}40\,\mathrm{Hz})}}{P_{\text{ruído} \,(<0.5\,\mathrm{Hz}\;\cup\;>40\,\mathrm{Hz})}}\right)$$

- **Sinal:** potência integrada na faixa **1–40 Hz** (onde vivem as ondas P, QRS e T).
- **Ruído:** potência integrada **fora** dessa faixa (baseline wander + EMG/PLI).
- Um **SNR maior** após a filtragem prova que a "voz" do coração ficou mais alta que o "chiado" do ambiente.

> 🎤 **Fala:** Matheo.

In [ ]:
from src.preprocessing.metrics import band_power

def snr_db(x: np.ndarray, fs: float,
           signal_band=(1.0, 40.0),
           noise_bands=((0.0, 0.5), (40.0, 180.0))) -> float:
    p_sig = band_power(x, fs, *signal_band)
    p_noi = sum(band_power(x, fs, lo, hi) for lo, hi in noise_bands)
    if p_noi <= 1e-20:
        return float("inf")
    return 10.0 * np.log10(p_sig / p_noi)

snr_before = snr_db(x_raw,  fs)
snr_after  = snr_db(x_filt, fs)

print(f"SNR (cru)      : {snr_before:6.2f} dB")
print(f"SNR (filtrado) : {snr_after:6.2f} dB")
print(f"Ganho de SNR   : {snr_after - snr_before:6.2f} dB")

## 8. Próximas Etapas — Roadmap do Projeto

Com a **Fase 1** consolidando o pipeline de denoising por FIR, o projeto avança para:

- **Fase 2 — Filtros de Gabor 1D.** Análise localizada por janelas Gaussianas moduladas, capazes de acompanhar variações tempo-frequência do ECG (útil em arritmias).
- **Fase 3 — Análise Tempo-Frequência.** Espectrogramas (STFT) e mapas tempo-frequência completos para caracterizar batimentos anômalos e transições rítmicas.
- **Fase 4 — Métricas Clínicas.** Detecção robusta de picos R, intervalos RR, variabilidade da frequência cardíaca (HRV) e classificação de batimentos sobre os 48 registos da MIT-BIH.

**Mensagem central da Fase 1:** o filtro FIR de fase linear é a base sobre a qual as próximas análises se apoiam — sem morfologia preservada, nenhuma métrica clínica é confiável.

> 🎤 **Fala:** Juliano (fechamento).